# DipRadar v2 — Training Notebook

**Objectivo:** Treinar e avaliar os modelos v2 comparando com produção v1.

**Fluxo:**
1. Setup (instalar deps + clonar repo — sem uploads necessários)
2. Carregar dataset de treino (`ml_training_merged.parquet` do repo)
3. Train/val split temporal (80/20)
4. Treinar v2 (XGBoost upside + downside)
5. Avaliar (Spearman, Top-K, comparação com v1)
6. Feature importance
7. Guardar modelos (download .pkl)

> 💡 **Nota sobre dados:** Este notebook usa o `ml_training_merged.parquet` do repositório.
> O retreino mensal em produção (Railway) incorpora o `alert_db.csv` automaticamente — não precisas de o fazer aqui.
> Quando quiseres incluir alertas reais neste notebook, basta descommentar a célula da Secção 2B.

> ⚠️ **Anti-leakage:** features e targets são calculados em janelas estritamente separadas.

## 1. Setup

In [ ]:
!pip install -q xgboost lightgbm yfinance scipy pyarrow scikit-learn

In [ ]:
import os

# Se o repo for privado, gera um token em GitHub → Settings → Developer Settings → Fine-grained tokens
GITHUB_TOKEN = ''  # @param {type:'string'}

if os.path.exists('DipRadar'):
    print('Repo já clonado — a fazer pull...')
    !cd DipRadar && git pull
elif GITHUB_TOKEN:
    !git clone https://{GITHUB_TOKEN}@github.com/romeurf/DipRadar.git
else:
    !git clone https://github.com/romeurf/DipRadar.git

os.chdir('DipRadar')
!pwd && ls

## 2. Carregar Dataset de Treino

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Dataset histórico do repo (gerado pelo bootstrap_ml.py)
PARQUET_PATH = Path('ml_training_merged.parquet')

df = pd.read_parquet(PARQUET_PATH)
df['alert_date'] = pd.to_datetime(df['alert_date'])
df = df.sort_values('alert_date').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Período: {df["alert_date"].min().date()} → {df["alert_date"].max().date()}')
print(f'Colunas: {list(df.columns)}')
print()
print('Distribuição de labels:')
if 'outcome_label' in df.columns:
    print(df['outcome_label'].value_counts())
elif 'label_win' in df.columns:
    print(f"  WIN: {df['label_win'].sum()} | LOSE: {(df['label_win']==0).sum()} | Win rate: {df['label_win'].mean():.1%}")
df.head(3)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2B. OPCIONAL — Incluir alert_db.csv com alertas reais
#
# O retreino mensal no Railway faz isto automaticamente.
# Aqui só precisas se quiseres forçar inclusão no Colab.
#
# Para obter o alert_db.csv: manda /exportdb no bot Telegram
# e faz upload abaixo.
# ─────────────────────────────────────────────────────────────

INCLUDE_ALERT_DB = False  # @param {type:'boolean'}

if INCLUDE_ALERT_DB:
    from google.colab import files
    print('Faz upload do alert_db.csv:')
    uploaded = files.upload()

    for fname in uploaded:
        df_alerts = pd.read_csv(fname)
        # Filtrar só rows com outcome resolvido
        if 'outcome_label' in df_alerts.columns:
            df_alerts = df_alerts[df_alerts['outcome_label'].notna() & (df_alerts['outcome_label'] != '')]
        if 'label_win' not in df_alerts.columns and 'outcome_label' in df_alerts.columns:
            df_alerts['label_win'] = df_alerts['outcome_label'].isin(['WIN_40','WIN_20']).astype(int)

        # Normalizar nome da coluna de data
        date_col = 'date_iso' if 'date_iso' in df_alerts.columns else 'alert_date'
        df_alerts = df_alerts.rename(columns={date_col: 'alert_date'})
        df_alerts['alert_date'] = pd.to_datetime(df_alerts['alert_date'])

        print(f'alert_db carregado: {len(df_alerts)} linhas com outcome resolvido')

        # Concatenar e deduplicar (alert_db tem prioridade sobre parquet)
        if 'symbol' not in df_alerts.columns and 'ticker' in df_alerts.columns:
            df_alerts = df_alerts.rename(columns={'ticker': 'symbol'})

        df['_src'] = 'parquet'
        df_alerts['_src'] = 'alert_db'

        # Colunas em comum
        common_cols = [c for c in df.columns if c in df_alerts.columns]
        df_merged = pd.concat([df[common_cols], df_alerts[common_cols]], ignore_index=True)
        df_merged = df_merged.sort_values('_src').drop_duplicates(
            subset=['symbol','alert_date'], keep='last'
        ).drop(columns=['_src'])
        df = df_merged.sort_values('alert_date').reset_index(drop=True)

        print(f'Dataset final após merge: {len(df)} linhas')
else:
    print('A usar só o ml_training_merged.parquet do repo. (INCLUDE_ALERT_DB=False)')

## 2C. Análise Exploratória Rápida

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição temporal
df['alert_date'].dt.year.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Amostras por ano')
axes[0].set_xlabel('')

# Target: return_6m ou label_win
target_col = 'return_6m' if 'return_6m' in df.columns else 'label_win'
df[target_col].dropna().hist(bins=30, ax=axes[1], color='mediumseagreen', edgecolor='white')
axes[1].set_title(f'Distribuição: {target_col}')

# Win rate ao longo do tempo (rolling)
if 'label_win' in df.columns:
    df['label_win'].rolling(200, min_periods=50).mean().plot(ax=axes[2], color='tomato')
    axes[2].set_title('Win rate rolling (200 amostras)')
    axes[2].axhline(df['label_win'].mean(), linestyle='--', color='black', alpha=0.5, label=f'média={df["label_win"].mean():.1%}')
    axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Train / Val Split (temporal — nunca aleatório)

In [ ]:
# Features disponíveis no parquet de produção
FEATURE_COLS = [
    'drawdown_52w', 'drop_pct_today', 'rsi_14', 'fcf_yield',
    'revenue_growth', 'gross_margin', 'de_ratio',
    'spy_drawdown_5d', 'sector_drawdown_5d',
    'macro_score', 'vix', 'atr_ratio', 'volume_spike',
    'pe_vs_fair', 'analyst_upside', 'quality_score',
]

# Usar apenas colunas que existem no dataset
feat_cols = [c for c in FEATURE_COLS if c in df.columns]
print(f'Features disponíveis ({len(feat_cols)}/{len(FEATURE_COLS)}): {feat_cols}')

missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f'⚠️  Colunas em falta (serão ignoradas): {missing}')

# Target
TARGET_COL = 'label_win'  # classificação binária (modelo v1 actual)

df_model = df[feat_cols + [TARGET_COL, 'alert_date']].dropna(subset=[TARGET_COL]).copy()
df_model[feat_cols] = df_model[feat_cols].fillna(df_model[feat_cols].median())

# Split 80/20 temporal
split_idx = int(len(df_model) * 0.80)
df_train  = df_model.iloc[:split_idx]
df_val    = df_model.iloc[split_idx:]

X_train = df_train[feat_cols].values
y_train = df_train[TARGET_COL].values
X_val   = df_val[feat_cols].values
y_val   = df_val[TARGET_COL].values

print(f'\nTrain : {len(df_train)} amostras | {df_train["alert_date"].min().date()} → {df_train["alert_date"].max().date()} | win_rate={y_train.mean():.1%}')
print(f'Val   : {len(df_val)}   amostras | {df_val["alert_date"].min().date()} → {df_val["alert_date"].max().date()} | win_rate={y_val.mean():.1%}')

## 4. Treinar Modelo v2

In [ ]:
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Calcular scale_pos_weight para lidar com desequilíbrio de classes
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos
print(f'Class balance — neg={neg}, pos={pos}, scale_pos_weight={spw:.2f}')

# Modelo v2 — XGBoost com hiperparâmetros refinados
model_v2 = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=spw,
    eval_metric='aucpr',
    early_stopping_rounds=30,
    random_state=42,
    verbosity=0,
)

model_v2.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f'\nMelhor iteração: {model_v2.best_iteration}')

## 5. Avaliar — Comparar v1 vs v2

In [ ]:
import pickle
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

# Predições v2
proba_v2 = model_v2.predict_proba(X_val)[:, 1]
auc_pr_v2  = average_precision_score(y_val, proba_v2)
auc_roc_v2 = roc_auc_score(y_val, proba_v2)

# Modelo v1 de produção (para comparação)
v1_path = Path('dip_model_stage1.pkl')
auc_pr_v1 = None
if v1_path.exists():
    try:
        with open(v1_path, 'rb') as f:
            model_v1 = pickle.load(f)
        # Tentar predict_proba — depende do tipo de modelo
        if hasattr(model_v1, 'predict_proba'):
            proba_v1 = model_v1.predict_proba(X_val)[:, 1]
        elif hasattr(model_v1, 'predict'):
            proba_v1 = model_v1.predict(X_val)
        else:
            proba_v1 = None

        if proba_v1 is not None:
            auc_pr_v1  = average_precision_score(y_val, proba_v1)
            auc_roc_v1 = roc_auc_score(y_val, proba_v1)
    except Exception as e:
        print(f'⚠️  Não foi possível carregar v1 para comparação: {e}')
else:
    print('ℹ️  dip_model_stage1.pkl não encontrado — a mostrar só métricas v2')

print('=' * 50)
print('           AUC-PR    AUC-ROC')
print('-' * 50)
if auc_pr_v1:
    delta = auc_pr_v2 - auc_pr_v1
    arrow = '↑' if delta > 0 else '↓'
    print(f'  v1 (prod)  {auc_pr_v1:.4f}    {auc_roc_v1:.4f}')
    print(f'  v2 (novo)  {auc_pr_v2:.4f}    {auc_roc_v2:.4f}   {arrow} {abs(delta):.4f}')
    print()
    if auc_pr_v2 >= auc_pr_v1 * 0.95:
        print('✅ v2 passa o gating (≥ v1 × 0.95) — pode ser promovido')
    else:
        print('❌ v2 não passa o gating — manter v1 em produção')
else:
    print(f'  v2 (novo)  {auc_pr_v2:.4f}    {auc_roc_v2:.4f}')
print('=' * 50)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva PR
prec_v2, rec_v2, _ = precision_recall_curve(y_val, proba_v2)
axes[0].plot(rec_v2, prec_v2, label=f'v2 AUC-PR={auc_pr_v2:.4f}', color='steelblue')
if auc_pr_v1:
    prec_v1, rec_v1, _ = precision_recall_curve(y_val, proba_v1)
    axes[0].plot(rec_v1, prec_v1, label=f'v1 AUC-PR={auc_pr_v1:.4f}', color='tomato', linestyle='--')
axes[0].axhline(y_val.mean(), linestyle=':', color='gray', label=f'baseline random={y_val.mean():.2%}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Histograma de probabilidades
axes[1].hist(proba_v2[y_val==0], bins=25, alpha=0.6, color='tomato', label='LOSE', density=True)
axes[1].hist(proba_v2[y_val==1], bins=25, alpha=0.6, color='steelblue', label='WIN', density=True)
axes[1].set_title('Distribuição de probabilidades v2')
axes[1].set_xlabel('P(WIN)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pr_curve_v2.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Top-K precision: se ordenas por P(WIN) descendente, qual o win rate no top 10% / 20%?
import pandas as pd

val_df = pd.DataFrame({'prob': proba_v2, 'win': y_val}).sort_values('prob', ascending=False)

print('Top-K precision (v2):')
for k in [0.05, 0.10, 0.20, 0.30]:
    n = max(1, int(len(val_df) * k))
    top = val_df.head(n)
    baseline = val_df['win'].mean()
    precision = top['win'].mean()
    lift = precision / baseline
    ok = '✅' if precision > baseline else '❌'
    print(f'  Top {k:.0%} (n={n:3d}): precision={precision:.1%}  baseline={baseline:.1%}  lift={lift:.2f}x  {ok}')

## 6. Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

imp = pd.Series(model_v2.feature_importances_, index=feat_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(feat_cols) * 0.35)))
imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance — XGBoost v2 (gain)')
ax.set_xlabel('Importance')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 5 features:')
print(imp.tail(5).sort_values(ascending=False).to_string())

## 7. Guardar e Fazer Download

In [ ]:
import pickle, json
from pathlib import Path

# Critérios de promoção (mesmo gating do Railway)
gating_ok = auc_pr_v1 is None or (auc_pr_v2 >= auc_pr_v1 * 0.95 and auc_pr_v2 >= 0.18)

print('Critérios de promoção:')
if auc_pr_v1:
    print(f'  AUC-PR v2 >= v1 × 0.95 : {"✅" if auc_pr_v2 >= auc_pr_v1 * 0.95 else "❌"} ({auc_pr_v2:.4f} >= {auc_pr_v1 * 0.95:.4f})')
print(f'  AUC-PR v2 >= floor 0.18 : {"✅" if auc_pr_v2 >= 0.18 else "❌"} ({auc_pr_v2:.4f})')
print()

if gating_ok:
    print('✅ Modelo pronto para ser promovido!')
else:
    print('❌ Modelo NÃO passa o gating — não substituir em produção ainda.')

# Guardar sempre para análise, mesmo que não passe o gating
output_dir = Path('experiments/ml_v2')
output_dir.mkdir(parents=True, exist_ok=True)

# Guardar modelo + metadados
bundle = {
    'model': model_v2,
    'feature_cols': feat_cols,
    'gating_ok': gating_ok,
    'auc_pr_v2': float(auc_pr_v2),
    'auc_pr_v1': float(auc_pr_v1) if auc_pr_v1 else None,
    'n_train': len(X_train),
    'n_val': len(X_val),
    'train_date': str(pd.Timestamp.now().date()),
}

pkl_path = output_dir / 'dip_model_v2_candidate.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(bundle, f)
print(f'Guardado: {pkl_path}')

# Report JSON
report = {k: v for k, v in bundle.items() if k != 'model'}
report_path = output_dir / 'eval_report_v2.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(f'Guardado: {report_path}')

In [ ]:
from google.colab import files

files.download(str(pkl_path))
files.download(str(report_path))
files.download('pr_curve_v2.png')
files.download('feature_importance_v2.png')
print('Downloads iniciados!')

## ✅ Próximos Passos

Se `gating_ok = True`:

1. **Fazer commit** do `dip_model_v2_candidate.pkl` para `experiments/ml_v2/`
2. Testar em **shadow mode**: o bot gera alertas com v1 mas loga também o score v2 para comparar
3. Só após 2–4 semanas de shadow, substituir `dip_model_stage1.pkl` em produção

**O retreino mensal automático no Railway continua a enriquecer o parquet com o `alert_db.csv` — não precisas de fazer nada para isso.**